# Using AI for spatial data analysis

In this notebook, I learned how to use SRAI (an AI library built for spatial analysis) to regionalize and embed data showing transmission lines in Colorado to create a map showing the density of transmission lines throughout Colorado. 

In [ ]:
#Use to install SRAI. May also want to run %pip install osm and %pip install osmnx
%pip install srai

In [ ]:
from srai.loaders import OSMOnlineLoader
from srai.plotting import plot_regions
from srai.regionalizers import geocode_to_region_gdf
import geopandas as gp
import pandas

#Loads data about transmission lines in Colorado from OpenStreetMap
query = {"power": "line"}
area = geocode_to_region_gdf("Colorado, USA")
loader = OSMOnlineLoader()
parks_gdf = loader.load(area, query)

#This was used to load HIFLD information, but as of right now only OSM data is consistently compatable with SRAI
'''parks_gdf = gp.read_file("Data/Electric_Power_Transmission_Lines.geojson")
nationalLinesBuffers = parks_gdf.buffer(0.043, resolution=16, cap_style='round', join_style='round', mitre_limit=5.0, single_sided=False)
nationalLinesBuffers.to_crs('EPSG:4326')
nationalBuffersGDF = gp.GeoDataFrame(nationalLinesBuffers, geometry=gp.GeoSeries(nationalLinesBuffers))
folium_map = plot_regions(area, colormap=["rgba(0,0,0,0)"], tiles_style="CartoDB positron")
nationalBuffersGDF.explore(m=folium_map, color="forestgreen")'''


In [ ]:
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf

#Regionalizes (or breaks up a region into smaller pieces) the Colorado region with a resolution of 5. Resolutions on this scale between 4-7 are recommended. 
regionalizer = H3Regionalizer(resolution=5)
regions = regionalizer.transform(area)

#Creates an interactive showing the regionalized data
folium_map = plot_regions(area, colormap=["rgba(0,0,0,0.1)"], tiles_style="CartoDB positron")
plot_regions(regions_gdf=regions, map=folium_map)

In [ ]:
from srai.embedders import CountEmbedder
from srai.joiners import IntersectionJoiner
from srai.plotting import plot_regions, plot_numeric_data

#Joins the transmission data to the regions created in the last cell
joiner = IntersectionJoiner()
joint = joiner.transform(regions, parks_gdf)

#Uses a basic embedder that doesn't require fitting to embed the transmission line data into the regionalized map to show the density of transmission lines in Colorado
embedder = CountEmbedder()
embeddings = embedder.transform(regions, parks_gdf, joint)

#Plots the embedded map created before
folium_map = plot_regions(area, colormap=["rgba(0,0,0,0.1)"], tiles_style="CartoDB positron")
plot_numeric_data(regions, "power_line", embeddings, map=folium_map)